## 1. Load and Import

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings("ignore", message="Could not parse column 'reversed'")

DATA_PROC = Path("../../data/processed")
DATA_EXT  = Path("../../data/external")
DATA_OUT  = Path("../../data/processed")

In [2]:
tracts    = gpd.read_file(DATA_PROC / "svi_tracts.geojson")
stations  = gpd.read_file(DATA_PROC / "all_stations.geojson")
hydrants  = gpd.read_file(DATA_PROC / "Fire_Hydrants.geojson")
hospitals = gpd.read_file(DATA_PROC / "hospitals.geojson")
roads     = gpd.read_file(DATA_EXT  / "escape_routes_roads.geojson")

# join key: "tract" (float64, e.g. 1011.1) — full FIPS was dropped during SVI processing
# RPL columns are snake_case: rpl_themes, rpl_theme_1..4
# 25 tracts have rpl_themes == NaN (originally -999 sentinels, replaced at processing time)

CRS = "EPSG:3310"  # CA Albers, meters — best for LA distance/area calculations
tracts    = tracts.to_crs(CRS)
stations  = stations.to_crs(CRS)
hydrants  = hydrants.to_crs(CRS)
hospitals = hospitals.to_crs(CRS)
roads     = roads.to_crs(CRS)

print(tracts.shape, stations.shape, hydrants.shape, hospitals.shape)
print("rpl_themes NaNs:", tracts["rpl_themes"].isna().sum())

(2493, 152) (412, 6) (60295, 3) (165, 4)
rpl_themes NaNs: 25


## Feature 1: Distance to nearest fire station (per tract centroid)

In [3]:
tract_centroids = tracts.copy()
tract_centroids["geometry"] = tracts.geometry.centroid

def nearest_distance(origins, destinations):
    """Returns array of distances (meters) from each origin to nearest destination."""
    dest_union = destinations.geometry.union_all()
    return origins.geometry.apply(
        lambda pt: pt.distance(nearest_points(pt, dest_union)[1])
    )

tracts["dist_fire_station_m"] = nearest_distance(tract_centroids, stations)
print(tracts["dist_fire_station_m"].describe())

count     2493.000000
mean      1337.833240
std       1747.598043
min         21.555872
25%        791.078160
50%       1177.750111
75%       1607.861827
max      62357.005818
Name: dist_fire_station_m, dtype: float64


## Feature 2: Distance to nearest hospital

In [4]:
tracts["dist_hospital_m"] = nearest_distance(tract_centroids, hospitals)
print(tracts["dist_hospital_m"].describe())

count     2493.000000
mean      2983.523795
std       2877.505335
min         61.365995
25%       1413.948820
50%       2401.161141
75%       3672.355071
max      37682.365498
Name: dist_hospital_m, dtype: float64


##  Feature 3: Hydrant density (hydrants per km² within tract)

In [5]:
hydrants_in_tracts = gpd.sjoin(
    hydrants,
    tracts[["tract", "geometry"]],
    how="inner",
    predicate="within"
)
hydrant_counts = hydrants_in_tracts.groupby("tract").size().rename("hydrant_count")

if "hydrant_count" in tracts.columns:
    tracts = tracts.drop(columns=["hydrant_count"])
tracts = tracts.merge(hydrant_counts, on="tract", how="left")
tracts["hydrant_count"] = tracts["hydrant_count"].fillna(0)
tracts["tract_area_km2"] = tracts.geometry.area / 1e6
tracts["hydrant_density"] = tracts["hydrant_count"] / tracts["tract_area_km2"]

print(tracts["hydrant_density"].describe())

count    2493.000000
mean       30.952924
std        38.187512
min         0.000000
25%         0.000000
50%         0.000000
75%        64.421846
max       261.355381
Name: hydrant_density, dtype: float64


## Feature 4: Road escape density (road length per km² within tract)


In [6]:
roads_clipped = gpd.overlay(
    roads.to_crs("EPSG:3310"),
    tracts[["tract", "geometry"]].to_crs("EPSG:3310"),
    how="intersection"
)
roads_clipped["length_m"] = roads_clipped.geometry.length

road_length_per_tract = (
    roads_clipped.groupby("tract")["length_m"]
    .sum()
    .rename("road_length_m")
)

tracts = tracts.to_crs("EPSG:3310")
tracts["tract_area_km2"] = tracts.geometry.area / 1e6
if "road_length_m" in tracts.columns:
    tracts = tracts.drop(columns=["road_length_m"])
tracts = tracts.merge(road_length_per_tract, on="tract", how="left")
tracts["road_length_m"] = tracts["road_length_m"].fillna(0)
tracts["road_density"] = tracts["road_length_m"] / tracts["tract_area_km2"]
print(tracts["road_density"].describe())

count     2493.000000
mean      7171.994281
std       4150.211696
min          0.000000
25%       4309.327072
50%       6683.852786
75%       9372.396721
max      32131.626521
Name: road_density, dtype: float64


tracts["tract_area_km2"].head(10)

## Feature 5: Population Density

In [7]:
tracts["e_totpop"].head(10)

0    3923
1    4119
2    3775
3    3787
4    2717
5    3741
6    3246
7    1920
8    4187
9    1797
Name: e_totpop, dtype: int32

In [8]:
POP_COL = "e_totpop"  # ⚠️ change if needed

if POP_COL in tracts.columns:
    tracts["pop_density"] = tracts[POP_COL] / tracts["tract_area_km2"]
else:
    print("WARNING: Population column not found — skipping pop_density")
    tracts["pop_density"] = 0

In [9]:
tracts["pop_density"].head(10)

0     3433.999192
1     1557.839218
2     5401.466419
3    10692.407128
4     9163.170797
5     1449.728335
6      514.416965
7     1618.226514
8     2524.490199
9     3646.799734
Name: pop_density, dtype: float64

### Normalize function

In [10]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

## Feature 6: Composite Risk Score

In [11]:
# --- NORMALIZED FEATURES ---
tracts["norm_dist_fire"] = normalize(tracts["dist_fire_station_m"])
tracts["norm_road_inv"]  = normalize(1 / (tracts["road_density"] + 1e-6))
tracts["norm_hyd_inv"]   = normalize(1 / (tracts["hydrant_density"] + 1e-6))
tracts["norm_svi"]       = normalize(tracts["rpl_themes"])

# --- COMPOSITE RISK SCORE ---
tracts["risk_score"] = (
    0.4 * tracts["norm_svi"] +
    0.3 * tracts["norm_dist_fire"] +
    0.2 * tracts["norm_road_inv"] +
    0.1 * tracts["norm_hyd_inv"]
)

## Risk Tier Labels

In [12]:
valid_mask = tracts["risk_score"].notna()

# Quantile bins on valid tracts → ~equal thirds for balanced ML classes
tracts.loc[valid_mask, "risk_tier"] = pd.qcut(
    tracts.loc[valid_mask, "risk_score"].rank(method="first"),
    q=3,
    labels=[0, 1, 2],   # 0 = Low, 1 = Medium, 2 = High
).astype(float)

# Tracts with NaN risk_score keep NaN risk_tier — dropped in model_trainning.ipynb
TIER_LABELS = {0: "Low", 1: "Medium", 2: "High"}
print(tracts["risk_tier"].value_counts().sort_index().rename(TIER_LABELS))
print(f"NaN risk_tier (no SVI data, dropped before training): {tracts['risk_tier'].isna().sum()}")

risk_tier
Low       823
Medium    822
High      823
Name: count, dtype: int64
NaN risk_tier (no SVI data, dropped before training): 25


## Save feature matrix

In [13]:
feature_cols = [
    "tract",
    "dist_fire_station_m",
    "dist_hospital_m",
    "hydrant_density",
    "road_density",
    "pop_density",
    "rpl_theme_1",   # Socioeconomic status
    "rpl_theme_2",   # Household characteristics
    "rpl_theme_3",   # Racial/ethnic minority status
    "rpl_theme_4",   # Housing type & transportation
    "risk_score",
    "risk_tier",
    "geometry",
]

features = tracts[feature_cols].copy()
print(features.shape)
print(features.isna().sum())
features.to_file(DATA_OUT / "features.geojson", driver="GeoJSON")
print("Saved to", DATA_OUT / "features.geojson")

(2493, 13)
tract                   0
dist_fire_station_m     0
dist_hospital_m         0
hydrant_density         0
road_density            0
pop_density             0
rpl_theme_1            23
rpl_theme_2            21
rpl_theme_3            17
rpl_theme_4            24
risk_score             25
risk_tier              25
geometry                0
dtype: int64
Saved to ../../data/processed/features.geojson
